# Лабораторная работа № 3. Часть 1. Побайтовое XOR-шифрование

**Вариант № 10.** Исходный текст и ключ взяты из столбцов B и C файла «ММБИС Варианты для лаб 3_1.xlsx». Нужно получить криптограмму в шестнадцатеричном виде и восстановить исходный текст.

При запуске из корня проекта исходный текст и ключ загружаются из `data/lab_03_1.json`.

## 1. Шифрование и дешифрование

Кодируем строки в CP1251, как в примере `xor_encode.txt`. Перед шифрованием убираем пробелы по краям открытого текста. Ключ повторяется циклически. Для байта сообщения с индексом $i$ используется байт ключа с индексом $i\bmod m$, где $m$ — длина ключа. Операция XOR обратима, поскольку $(a\oplus b)\oplus b=a$.

In [1]:
ENCODING = 'cp1251'

def xor_bytes(data, key):
    if not key:
        raise ValueError('Ключ должен содержать хотя бы один байт')
    return bytes(value ^ key[i % len(key)] for i, value in enumerate(data))

def encode(text, key):
    return xor_bytes(text.encode(ENCODING), key.encode(ENCODING)).hex()

def decode(ciphertext_hex, key):
    ciphertext = bytes.fromhex(ciphertext_hex)
    return xor_bytes(ciphertext, key.encode(ENCODING)).decode(ENCODING)

## 2. Расчёт по варианту

Каждый байт криптограммы записывается двумя шестнадцатеричными цифрами. Дешифратор сначала восстанавливает байты из hex, затем повторяет XOR с тем же ключом.

In [2]:
import json
from pathlib import Path

inputs = json.loads(Path('data/lab_03_1.json').read_text(encoding='utf-8'))
full_name = inputs['text'].strip()
key = inputs['key']

ciphertext_hex = encode(full_name, key)
restored = decode(ciphertext_hex, key)

print('Исходный текст —', full_name)
print('Ключ —', key)
print('Криптограмма (hex) —', ciphertext_hex)
print('Дешифрованный текст —', restored)

assert len(ciphertext_hex) == 2 * len(full_name.encode(ENCODING))
assert restored == full_name
assert decode(encode('Ёж', 'к'), 'к') == 'Ёж'
assert decode(encode('', 'к'), 'к') == ''

Исходный текст — Зелёный Артём Сергеевич
Ключ — снкшз
Криптограмма (hex) — 360801400a0a04ca3817035506d836141d091d0213051d
Дешифрованный текст — Зелёный Артём Сергеевич


## Вывод

Криптограмма получена побайтовым XOR и преобразована в hex. Обратное преобразование восстановило исходное ФИО с точностью до каждого символа.